In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv("data.csv")
print("Shape:", df.shape)
print("Missing values:\n", df.isna().sum())
print("Target distribution:\n", df['target'].value_counts(normalize=True))
print("Data types:\n", df.dtypes.value_counts())
print(df.describe())
sns.countplot(x='target', data=df)
plt.show()

X = df.drop(columns=['target'])
y = df['target']
num_features = X.select_dtypes(include=['int64', 'float64']).columns
cat_features = X.select_dtypes(include=['object', 'category']).columns

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_features),
    ('cat', cat_pipeline, cat_features)
])

models = {
    "RandomForest": RandomForestClassifier(n_estimators=200, random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
    "LightGBM": LGBMClassifier(random_state=42)
}

X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.2, random_state=42
)

def build_pipeline(model):
    return ImbPipeline(steps=[
        ('preprocess', preprocessor),
        ('balance', SMOTE(random_state=42)),
        ('model', model)
    ])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for name, model in models.items():
    pipe = build_pipeline(model)
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='f1_macro')
    print(f"{name}: Mean F1 = {scores.mean():.4f}")

base_learners = [
    ('rf', RandomForestClassifier(n_estimators=200, random_state=42)),
    ('xgb', XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)),
    ('lgb', LGBMClassifier(random_state=42))
]

stack_model = StackingClassifier(
    estimators=base_learners,
    final_estimator=GradientBoostingClassifier(random_state=42),
    cv=5
)

stack_pipe = build_pipeline(stack_model)
stack_pipe.fit(X_train, y_train)

y_pred = stack_pipe.predict(X_test)
y_proba = stack_pipe.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC Score:", roc_auc_score(y_test, y_proba))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues')
plt.show()


#2
import pandas as pd

df = pd.read_csv("data.csv")
print("Shape:", df.shape)
print(df.head())
print(df.info())

print("Missing values:\n", df.isnull().sum())
sns.heatmap(df.isnull(), cbar=False)
plt.title("Missing Value Heatmap")
plt.show()

df['target'].value_counts(normalize=True).plot(kind='bar')
plt.title("Target Distribution")
plt.show()

num_features = df.select_dtypes(include=['int64', 'float64']).columns
cat_features = df.select_dtypes(include=['object', 'category']).columns

print("Numeric features:", num_features.tolist())
print("Categorical features:", cat_features.tolist())

